# Final Feature Merge and Audit

This notebook combines all six feature tables (application, bureau, previous application, instalments, credit card, and POS/cash) into one final applicant-level dataset. It checks for overlapping feature names, tracks which sources are available for each applicant, and applies one last missingness check before saving the modelling dataset.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
feature_folder = project_root / "data" / "features"
application_path = feature_folder / "application_features.pkl"
bureau_path = feature_folder / "bureau_features.pkl"
previous_path = feature_folder / "previous_application_features.pkl"
installment_path = feature_folder / "installment_payment_features.pkl"
credit_card_path = feature_folder / "credit_card_features.pkl"
pos_path = feature_folder / "pos_cash_features.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "modeling" / "final_feature_dataset.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
required_paths = [application_path, bureau_path, previous_path, installment_path, credit_card_path, pos_path, training_ids_path, test_ids_path]
for required_path in required_paths:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Final output:", output_path)

Final output: /Users/taranveersingh/A-MRP/data/modeling/final_feature_dataset.pkl


## Load all applicant-level feature tables


In [7]:
application_features = pd.read_pickle(application_path)
bureau_features = pd.read_pickle(bureau_path)
previous_features = pd.read_pickle(previous_path)
installment_features = pd.read_pickle(installment_path)
credit_card_features = pd.read_pickle(credit_card_path)
pos_features = pd.read_pickle(pos_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)

source_shapes = pd.DataFrame([
    {"source": "application", "rows": len(application_features), "columns": application_features.shape[1]},
    {"source": "bureau", "rows": len(bureau_features), "columns": bureau_features.shape[1]},
    {"source": "previous_application", "rows": len(previous_features), "columns": previous_features.shape[1]},
    {"source": "installments", "rows": len(installment_features), "columns": installment_features.shape[1]},
    {"source": "credit_card", "rows": len(credit_card_features), "columns": credit_card_features.shape[1]},
    {"source": "pos_cash", "rows": len(pos_features), "columns": pos_features.shape[1]},
])
source_shapes

,source,rows,columns
0,application,307511,110
1,bureau,263491,51
2,previous_application,291057,54
3,installments,291643,38
4,credit_card,307511,43
5,pos_cash,307511,37


Each of these tables was built separately in the earlier feature-engineering notebooks. The application table has one row for every applicant, since it comes from the full population, while the others cover only the applicants who actually have that kind of history.


## Validate source keys and feature-name uniqueness


In [10]:
source_tables = {
    "application": application_features,
    "bureau": bureau_features,
    "previous_application": previous_features,
    "installments": installment_features,
    "credit_card": credit_card_features,
    "pos_cash": pos_features,
}
for source_name, table in source_tables.items():
    assert "SK_ID_CURR" in table.columns, f"Applicant ID missing from {source_name}."
    assert table["SK_ID_CURR"].is_unique, f"Applicant IDs are not unique in {source_name}."

seen_features = set(application_features.columns)
overlap_rows = []
for source_name, table in list(source_tables.items())[1:]:
    source_columns = set(table.columns) - {"SK_ID_CURR"}
    overlaps = sorted(source_columns.intersection(seen_features))
    overlap_rows.append({"source": source_name, "overlapping_features": len(overlaps), "names": overlaps})
    seen_features.update(source_columns)
feature_overlap_audit = pd.DataFrame(overlap_rows)
assert feature_overlap_audit["overlapping_features"].sum() == 0, "Feature names overlap across source tables."
feature_overlap_audit

,source,overlapping_features,names
0,bureau,0,[]
1,previous_application,0,[]
2,installments,0,[]
3,credit_card,0,[]
4,pos_cash,0,[]


No feature names overlap between any of the six tables, so nothing gets accidentally overwritten during the merge.


## Merge incomplete-history sources


In [13]:
def merge_history_source(base, source, availability_name):
    source_columns = [c for c in source.columns if c != "SK_ID_CURR"]
    available_ids = set(source["SK_ID_CURR"])
    base[availability_name] = base["SK_ID_CURR"].isin(available_ids).astype("int8")
    base = base.merge(source, on="SK_ID_CURR", how="left", validate="one_to_one")
    absent_mask = base[availability_name].eq(0)
    base.loc[absent_mask, source_columns] = base.loc[absent_mask, source_columns].fillna(0)
    return base

final_data = application_features.copy()
final_data = merge_history_source(final_data, bureau_features, "BUREAU_HISTORY_AVAILABLE")
final_data = merge_history_source(final_data, previous_features, "PREV_HISTORY_AVAILABLE")
final_data = merge_history_source(final_data, installment_features, "INST_HISTORY_AVAILABLE")
print("Shape after bureau, previous and installment merges:", final_data.shape)

Shape after bureau, previous and installment merges: (307511, 253)


Bureau, previous-application, and instalment history are merged in here. For applicants who do not have that kind of history, the new feature values are filled with 0 instead of left missing, and a flag column records whether the history was actually available.


## Merge credit-card and POS/cash sources


In [16]:
final_data = final_data.merge(credit_card_features, on="SK_ID_CURR", how="left", validate="one_to_one")
final_data = final_data.merge(pos_features, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Merged shape before final audit features:", final_data.shape)
print("Duplicate applicant IDs:", int(final_data["SK_ID_CURR"].duplicated().sum()))

Merged shape before final audit features: (307511, 331)
Duplicate applicant IDs: 0


Same approach for credit-card and POS/cash, these already came with their own history-available flag and zero-filling built in from the feature-engineering notebooks.


## Audit source availability


In [19]:
availability_columns = [
    "BUREAU_HISTORY_AVAILABLE", "PREV_HISTORY_AVAILABLE", "INST_HISTORY_AVAILABLE",
    "CC_HISTORY_AVAILABLE", "POS_HISTORY_AVAILABLE"
]
availability_audit = pd.DataFrame([
    {
        "source": column.replace("_HISTORY_AVAILABLE", ""),
        "available_applicants": int(final_data[column].sum()),
        "unavailable_applicants": int(final_data[column].eq(0).sum()),
        "availability_rate": final_data[column].mean(),
    }
    for column in availability_columns
])
availability_audit.round(5)

,source,available_applicants,unavailable_applicants,availability_rate
0,BUREAU,263491,44020,0.85685
1,PREV,291057,16454,0.94649
2,INST,291643,15868,0.94840
3,CC,86905,220606,0.28261
4,POS,289444,18067,0.94125


Bureau, previous-application, instalment, and POS/cash history are all available for at least 85% of applicants. Credit-card history is the exception, only about 28% of applicants have any credit-card records at all.


## Add final row-level missingness features


In [22]:
predictors_before_missingness = [c for c in final_data.columns if c not in ["SK_ID_CURR", "TARGET"]]
final_data["FINAL_MISSING_COUNT"] = final_data[predictors_before_missingness].isna().sum(axis=1).astype("int16")
final_data["FINAL_MISSING_RATE"] = final_data["FINAL_MISSING_COUNT"] / len(predictors_before_missingness)
print(final_data[["FINAL_MISSING_COUNT", "FINAL_MISSING_RATE"]].describe().round(5))

       FINAL_MISSING_COUNT  FINAL_MISSING_RATE
count         307511.00000        307511.00000
mean               7.78751             0.02367
std                6.07524             0.01847
min                0.00000             0.00000
25%                3.00000             0.00912
50%                7.00000             0.02128
75%               11.00000             0.03343
max               44.00000             0.13374


/var/folders/wf/l8krgk7d6b39b_42bz8n2mdh0000gn/T/ipykernel_16920/4145881648.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_data["FINAL_MISSING_COUNT"] = final_data[predictors_before_missingness].isna().sum(axis=1).astype("int16")
/var/folders/wf/l8krgk7d6b39b_42bz8n2mdh0000gn/T/ipykernel_16920/4145881648.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_data["FINAL_MISSING_RATE"] = final_data["FINAL_MISSING_COUNT"] / len(predictors_before_missingness)


This counts how many predictor values are still missing for each applicant, on average about 2% of their fields.


## Apply final training-only missingness and constant-feature rules


In [25]:
MISSING_THRESHOLD = 0.50
training_data = final_data.loc[final_data["SK_ID_CURR"].isin(training_id_set)].copy()
decision_rows = []
for feature in [c for c in final_data.columns if c not in ["SK_ID_CURR", "TARGET"]]:
    series = training_data[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    is_numeric = pd.api.types.is_numeric_dtype(series)
    correlation = series.corr(training_data["TARGET"]) if is_numeric and unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for cross-validated model-based feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set"
    decision_rows.append({
        "feature": feature, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
final_feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = final_feature_decisions.loc[final_feature_decisions["decision"] == "Remove", "feature"].tolist()
final_data = final_data.drop(columns=removed_features)
print("Final features removed:", removed_features)
print("Final predictors retained:", final_data.shape[1] - 2)

Final features removed: []
Final predictors retained: 331


No features were removed at this final check. Everything from the individual feature-engineering notebooks already passed the missing-value and constant-value limits, so nothing new gets dropped here.


## Build the final feature dictionary


In [28]:
CLEANING_STAGE_ENGINEERED_COLUMNS = {
    "EXT_SOURCE_MEAN", "EXT_SOURCE_COUNT", "EXT_SOURCE_ALL_MISSING",
    "APPLICATION_MISSING_COUNT", "APPLICATION_MISSING_RATE",
}

def identify_source(feature):
    if feature in CLEANING_STAGE_ENGINEERED_COLUMNS:
        return "application_engineered"
    if feature.startswith("BUREAU_"):
        return "bureau"
    if feature.startswith("PREV_"):
        return "previous_application"
    if feature.startswith("INST_"):
        return "installments"
    if feature.startswith("CC_"):
        return "credit_card"
    if feature.startswith("POS_"):
        return "pos_cash"
    if feature.startswith("APP_"):
        return "application_engineered"
    if feature.startswith("FINAL_"):
        return "final_audit"
    return "application_original"

feature_dictionary = pd.DataFrame([
    {
        "feature": feature,
        "source": identify_source(feature),
        "data_type": str(final_data[feature].dtype),
        "missing_count_full": int(final_data[feature].isna().sum()),
        "missing_rate_full": final_data[feature].isna().mean(),
        "unique_non_missing_full": int(final_data[feature].nunique(dropna=True)),
    }
    for feature in final_data.columns
])
feature_dictionary["source"].value_counts()

source
application_original      72
previous_application      54
bureau                    51
credit_card               42
application_engineered    38
installments              38
pos_cash                  36
final_audit                2
Name: count, dtype: int64

This tags every feature with which source table it came from, which makes it easier to trace where any given feature originated later on. Five columns (EXT_SOURCE_MEAN, EXT_SOURCE_COUNT, EXT_SOURCE_ALL_MISSING, APPLICATION_MISSING_COUNT, APPLICATION_MISSING_RATE) are labelled as engineered here, even though their names don't start with APP_, because they were actually created during the application-cleaning notebook rather than being part of the original Kaggle file.


## Validate the final modelling dataset


In [31]:
numeric_columns = final_data.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(final_data[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = final_feature_decisions.loc[final_feature_decisions["decision"] == "Keep"]
categorical_missing = int(final_data.select_dtypes(exclude="number").isna().sum().sum())
validation_checks = pd.DataFrame([
    {"check": "Application row count preserved", "passed": len(final_data) == len(application_features)},
    {"check": "One row per applicant", "passed": final_data["SK_ID_CURR"].is_unique},
    {"check": "Applicant IDs unchanged", "passed": final_data["SK_ID_CURR"].equals(application_features["SK_ID_CURR"])},
    {"check": "TARGET unchanged", "passed": final_data["TARGET"].equals(application_features["TARGET"])},
    {"check": "Training and test remain separate", "passed": len(training_id_set.intersection(test_id_set)) == 0},
    {"check": "All rows assigned to training or test set", "passed": len(training_ids) + len(test_ids) == len(final_data)},
    {"check": "No duplicated feature names", "passed": not final_data.columns.duplicated().any()},
    {"check": "No categorical missing values", "passed": categorical_missing == 0},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from final feature decisions", "passed": not training_data["SK_ID_CURR"].isin(test_id_set).any()},
])
assert validation_checks["passed"].all(), "At least one final feature-merge check failed."
validation_checks

,check,passed
0,Application row count preserved,True
1,One row per applicant,True
2,Applicant IDs unchanged,True
3,TARGET unchanged,True
4,Training and test remain separate,True
5,All rows assigned to training or test set,True
6,No duplicated feature names,True
7,No categorical missing values,True
8,No infinite numerical values,True
9,No retained feature reaches 50 percent trainin...,True


All checks passed.


## Save the final dataset and audit reports


In [34]:
final_data.to_pickle(output_path)
source_shapes.to_csv(audit_folder / "final_merge_source_shapes.csv", index=False)
feature_overlap_audit.to_csv(audit_folder / "final_merge_feature_overlap.csv", index=False)
availability_audit.to_csv(audit_folder / "final_merge_source_availability.csv", index=False)
final_feature_decisions.to_csv(audit_folder / "final_feature_decisions.csv", index=False)
feature_dictionary.to_csv(audit_folder / "final_feature_dictionary.csv", index=False)
validation_checks.to_csv(audit_folder / "final_feature_merge_validation.csv", index=False)
print("Final feature dataset saved:", output_path)
print("Output rows:", len(final_data))
print("Output columns:", final_data.shape[1])
print("Predictors retained:", final_data.shape[1] - 2)
print("Categorical predictors:", len([c for c in final_data.select_dtypes(exclude="number").columns if c not in ["SK_ID_CURR", "TARGET"]]))
print("Numerical predictors:", len([c for c in final_data.select_dtypes(include="number").columns if c not in ["SK_ID_CURR", "TARGET"]]))
print("Remaining numerical missing values:", int(final_data.select_dtypes(include="number").isna().sum().sum()))

Final feature dataset saved: /Users/taranveersingh/A-MRP/data/modeling/final_feature_dataset.pkl
Output rows: 307511
Output columns: 333
Predictors retained: 331
Categorical predictors: 13
Numerical predictors: 318
Remaining numerical missing values: 2394745


## Main results

This notebook merged all six feature tables into one final dataset with 331 predictors (318 numerical and 13 categorical) for all 307,511 applicants. No feature names overlapped, and no features needed to be removed at this stage.

Bureau, previous-application, instalment, and POS/cash history are available for most applicants, while credit-card history is only available for about 28%. Applicants without a given type of history have those features filled with 0, with a flag column marking which sources were actually available.

This final dataset is what the modelling notebooks build on. CODE_GENDER is kept here for later fairness checks, but will be excluded from the model's predictors.
